In [24]:
#实战加利福尼亚房价预测
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
import numpy as np

california_housing = fetch_california_housing(data_home='./data')
X, y = california_housing.data, california_housing.target

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

class CaliforniaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx].unsqueeze(0)
    
train_dataset = CaliforniaDataset(X_train_scaled, y_train)
val_dataset = CaliforniaDataset(X_val_scaled, y_val)
test_dataset = CaliforniaDataset(X_test_scaled, y_test)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"数据集大小: 总计 {len(X)}")
print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")
print(f"测试集: {len(test_dataset)} 样本")
print(f"特征维度: {X.shape[1]}")

print(f"\n批量情况演示:")
y_batch = torch.tensor([1.2, 2.3, 3.4])
print(f"批量标签原始形状: {y_batch.shape}")
y_batch_unsqueezed = y_batch.unsqueeze(1)
print(f"批量标签unsqueeze(1)后形状: {y_batch_unsqueezed.shape}")
print(f"批量标签内容:\n{y_batch_unsqueezed}")

for x,y in train_loader:
    print(x.shape)
    print(y.shape)
    break

数据集大小: 总计 20640
训练集: 14448 样本
验证集: 3096 样本
测试集: 3096 样本
特征维度: 8

批量情况演示:
批量标签原始形状: torch.Size([3])
批量标签unsqueeze(1)后形状: torch.Size([3, 1])
批量标签内容:
tensor([[1.2000],
        [2.3000],
        [3.4000]])
torch.Size([64, 8])
torch.Size([64, 1])


In [25]:
class HousePriceModel(nn.Module):
    def __init__(self, input_size, output_size=1, hidden_size=30):
        super(HousePriceModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
    
input_size = X.shape[1]
model = HousePriceModel(input_size=input_size, output_size=1, hidden_size=30)

print(model)
print(sum(p.numel() for p in model.parameters()))

HousePriceModel(
  (fc1): Linear(in_features=8, out_features=30, bias=True)
  (fc2): Linear(in_features=30, out_features=1, bias=True)
  (relu): ReLU()
)
301


In [26]:
from my_train_pro1 import Trainer, EarlyStopping, ModelCheckpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

early_stopping = EarlyStopping(
    patience=10,
    min_delta=0.001,
    mode='min'
)

model_checkpoint = ModelCheckpoint(
    filepath='./checkpoints/regression_model_epoch_{epoch}.ckpt',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    min_delta=0.001
)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=50,
    early_stopping=early_stopping,
    model_checkpoint=model_checkpoint
)

print("开始训练...")
trainer.train_regression(num_epochs=100)


使用设备: cuda
开始训练...
[Step 50] Val Loss: 3.5534
[Step 100] Val Loss: 2.0272
[Step 150] Val Loss: 1.1564
[Step 200] Val Loss: 0.9081
Epoch [1/100]  Train Loss: 2.3151
[Step 250] Val Loss: 0.8435
[Step 300] Val Loss: 0.8013
[Step 350] Val Loss: 0.7626
[Step 400] Val Loss: 0.7307
[Step 450] Val Loss: 0.7003
Epoch [2/100]  Train Loss: 0.7647
[Step 500] Val Loss: 0.6704
[Step 550] Val Loss: 0.6382
[Step 600] Val Loss: 0.6075
[Step 650] Val Loss: 0.5810
Epoch [3/100]  Train Loss: 0.6164
[Step 700] Val Loss: 0.5596
[Step 750] Val Loss: 0.5405
[Step 800] Val Loss: 0.5238
[Step 850] Val Loss: 0.5088
[Step 900] Val Loss: 0.4985
Epoch [4/100]  Train Loss: 0.5143
[Step 950] Val Loss: 0.4875
[Step 1000] Val Loss: 0.4799
[Step 1050] Val Loss: 0.4732
[Step 1100] Val Loss: 0.4666
Epoch [5/100]  Train Loss: 0.4647
[Step 1150] Val Loss: 0.4617
[Step 1200] Val Loss: 0.4578
[Step 1250] Val Loss: 0.4551
[Step 1300] Val Loss: 0.4492
[Step 1350] Val Loss: 0.4460
Epoch [6/100]  Train Loss: 0.4412
[Step 1400] Va